# 딥러닝 TTS 모델 🐶Bark를 이용해 말하는 AI 만들어보기
---
* 이 글은 깃허브의 [suno-ai/bark](https://github.com/suno-ai/bark) 문서를 번역 및 재구성해 작성했습니다. 생성된 결과물 또한 해당 URL에서 들어볼 수 있습니다.

흔히들 TTS라고 말하는 Text-to-Speech 모델이 있습니다. 이 모델은 글자로 적혀 있는 말은 사람이 말하는 것과 같이 음성 합성을 해 주는 모델인데요, 옛날에는 청각장애인을 위한 보조 기능에 머물러 있었지만 지금은 사용 범위가 확장되었습니다. 예를 들어, 화면이 없는 IoT 기기에서 음성 피드백을 주거나 AI 비서 서비스들의 상호작용에 이용되곤 하지요. 이번에는 파이썬(Python)을 이용해서, Suno AI에서 제공하는 오픈소스 딥러닝 TTS 모델인 🐶Bark를 이용해 보는 방법을 알아보겠습니다.  
</p></br></br>

## 이용하는 패키지
---
🐶Bark는 🤗HuggingFace TransFormers를 이용하는 추상화된 형태로 제공되며, 자체 패키지로도 제공됩니다. 저는 파이썬 bark 패키지를 이용해서 TTS 기능을 구현해 보았습니다. 이외에는 음성 파일을 저장하는데 쓸 scipy, jupyter 개발환경일 경우 화면에서 음성 파일과 상호작용할 수 있도록 해 주는 IPython.display 모듈을 추가로 이용해 보도록 하겠습니다.</p></br></br>

만약 Bark를 설치하지 않았다면, 아래 터미널 명령어를 사용해서 설치해 주시기 바랍니다. `pip install bark` 명령어로 설치되는 패키지의 경우, Suno AI에서 공식 지원하지 않는 패키지라고 하니 아래 명령어를 이용하는 것을 권장합니다.  
</p></br></br>

```bash
pip install git+https://github.com/suno-ai/bark.git
```  
</p></br></br>

## 코드 살펴보기
---
`bark.preload_models()` 함수를 이용하면 모델 로드 작업이 자동으로 이루어지며, 이 때 별도의 매개변수 입력이 없이도 동작합니다. 실질적인 생성 작업은 `bark.generate_audio()` 함수이며, 여기에 입력하는 텍스트 및 매개변수에 따라서 생성된 음성의 스타일이 결정됩니다. 이 모델의 특별한 점으로는, 다양한 언어를 자동으로 인식한다는 점과 노래를 즉석에서 만들어줄 수도 있다는 점입니다.  
</p>

만약, 음성 데이터를 생성할 때 GPU 가속을 이용하고 싶다면 12GB 이상의 VRAM을 확보해 두시기 바랍니다. 8GB VRAM 최소사양을 보여주는 small model도 있으니 미묘한 성능의 그래픽카드라면 이 쪽을 고민해 보셔도 좋아요. 그래픽카드의 성능이 아주 낮거나 별도의 외장그래픽카드가 없는 경우에는, 속도가 느리긴 하지만 아예 CPU로 실행해도 좋겠습니다. 이번 예제에서는 CPU 버전을 기준으로 실행해 봤어요.  
</p></br></br>

### 단순 문장을 읽는 TTS 이용해보기
---
단순 문장을 읽어주는 TTS는 `bark.generate_audio()` 함수를 단독으로 사용하면 됩니다. 여기 입력할 매개변수는 텍스트 프롬프트 정보 밖에 없어요. 해당 모델의 동작 예시에는 영어가 대표로 나와 있고, 영어 음성의 생성 품질이 가장 좋습니다. 하지만 만약 영어 이외의 언어를 입력한다고 해도 그냥 마음대로 적어넣기만 하면 제법 좋은 품질의 음성이 생성됩니다. 생성된 오디오 객체는 `scipy.io.wavfile.write.write_wav()` 함수를 이용해 wav 파일로 저장하거나, `IPython.display.Audio` 클래스를 이용해서 화면상에 보이도록 설정할 수도 있지요.  
</p></br></br>


In [1]:
# 경고 메시지 출력을 막기 위한 코드
import warnings
warnings.filterwarnings('ignore')

# Import Package
from bark import generate_audio, preload_models
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio

sample_rate = 24000

# download and load models
preload_models()

# generate audio from text
text_prompt = """
     Hello, my name is Suno. And, uh — and I like pizza. [laughs] 
     But I also have other interests such as playing tic tac toe.
"""

audio_array = generate_audio(text_prompt)

# save audio to disk
write_wav(filename="bark_generation.wav", rate=sample_rate, data=audio_array)
  
# play audio in notebook
Audio(data=audio_array, rate=sample_rate)

No GPU being used. Careful, inference might be very slow!
100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [07:01<00:00, 14.04s/it]


</p></br></br>

### 다양한 옵션으로 음성 및 음악 생성하기
---
위에서 알려드렸듯이, 🐶Bark는 기본적으로 영어 문장을 읽어주는 TTS 모델입니다. 하지만 13개 언어를 지원하는 모델답게 한국어를 포함한 다른 언어를 입력해도 곧잘 읽어주는데요, 별다른 설정 없이 해당 언어를 집어넣기만 하면 됩니다. 지원하는 언어 리스트는 [공식 깃허브 문서](https://github.com/suno-ai/bark#supported-languages)를 참조해 주시기 바랍니다.  
</p></br></br>

만약 단순히 문장을 읽어주는것 말고, 음악을 생성하고 싶다면 텍스트 프롬프트 앞뒤로 음표(♪)를 넣어 주면 됩니다. 이렇게 하면 🐶Bark가 자동으로 음악을 생성해 준다고 하며, 음성 프리셋을 이용할 경우 원하는 성우를 선택하듯이 다채로운 음성 생성이 가능하다고 합니다. 음성 프리셋은 [공식 노션 페이지](https://suno-ai.notion.site/8b8e8749ed514b0cbf3f699013548683?v=bc67cff786b04b50b3ceb756fd05f68c)에서 선택하시면 됩니다. 아래 코드는 한국어 문장을 읽는 코드와 음악을 생성하는 코드의 예시입니다.  
</p></br></br>


In [2]:
# 한국어 읽기
text_prompt = """
    추석은 내가 가장 좋아하는 명절이다. 나는 며칠 동안 휴식을 취하고 친구 및 가족과 시간을 보낼 수 있습니다.
"""
audio_array = generate_audio(text_prompt)

# play audio in notebook
Audio(data=audio_array, rate=sample_rate)

100%|██████████████████████████████████████████████████████████████████████████████████| 34/34 [07:41<00:00, 13.59s/it]


In [8]:
# 한국어 음성 프리셋 설정
text_prompt = """
    추석은 내가 가장 좋아하는 명절이다. 나는 며칠 동안 휴식을 취하고 친구 및 가족과 시간을 보낼 수 있습니다.
"""
audio_array = generate_audio(text_prompt, history_prompt="v2/ko_speaker_6")

# play audio in notebook
Audio(data=audio_array, rate=sample_rate)

100%|██████████████████████████████████████████████████████████████████████████████████| 35/35 [07:59<00:00, 13.69s/it]


In [3]:
# 음악 만들기
text_prompt = """
    ♪ In the jungle, the mighty jungle, the lion barks tonight ♪
"""
audio_array = generate_audio(text_prompt)

# play audio in notebook
Audio(data=audio_array, rate=sample_rate)

100%|██████████████████████████████████████████████████████████████████████████████████| 21/21 [04:16<00:00, 12.24s/it]


</p></br></br>

이외에도 다양한 음성 생성을 🐶Bark TTS 모델을 활용해 시도해볼 수 있습니다.